# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [117]:
#from langchain_community.document_loaders import WebBaseLoader
#loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")

from langchain_community.document_loaders import PyPDFLoader
file_path = "../ai_report_2025.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()
print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text)
#extracts the document text


26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agr

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [118]:
from pydantic import BaseModel
from openai import OpenAI
#import classes

client = OpenAI()
model_type="gpt-4o-mini" #change the model type here

class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int
#define the Pydantic Basemodel object for Article Summary, string for all except token counts which are int

tone = "Formal Academic Writing"
article_topic="technical AI content"
system_prompt = f"""
    You are an expert AI document summarization assistant specializing in {article_topic}.
    Your task is to summarize the provided document exactly as written, without adding, inferring, or interpreting any information beyond what is explicitly stated.  

    Instructions:  
    - Extract the title and author names exactly as they appear in the document.  
    - Quote directly from the source material wherever appropriate.  
    - Do not add, infer, or assume any information that is not explicitly stated in the document.  
    - Provide a relevance statement (maximum one paragraph) explaining why this article is important for the professional development of AI practitioners.  
    - Ensure the summary maintains a clear, logical structure and consistent tone: {tone}.  
    - Maximum output: 1000 tokens.

    Accuracy, fidelity to the source, and clarity are essential.
    Avoid introducing personal opinions, external knowledge, or unsupported claims.
    Every piece of information in the summary must be verifiable in the original text.
"""
user_prompt = f"""
    Please create a professional, concise summary of the following article, using only information explicitly stated in the text.  
    Instructions:  
    - Do not add, infer, or assume any information beyond what is present in the article.  
    - Only include facts, statements, and quotes found in the document.  
    - Maintain a clear and logical structure in your summary.  
    - Avoid paraphrasing in a way that changes the meaning of the original content.  

    The article to summarize is the following:\n
    <article>
    {document_text}
    </article>
"""
#prompt engineering

def generate_summary(system_prompt, user_prompt):
    response = client.responses.parse(
        model=model_type,
        input=[ #define the prompts
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ArticleSummary,#ensure the results are in correct format
    )
    return response.output_parsed #parse for Pydantic object
#define function that uses OpenAI API to generate summary
#def makes it easy to reuse this code later

article_summary = generate_summary(system_prompt, user_prompt) #define as a variable for use for later
print(f"Author: {article_summary.author}\n")
print(f"Title: {article_summary.title}\n")
print(f"Relevance: {article_summary.relevance}\n")
print(f"Summary: {article_summary.summary}\n")
print(f"Tone: {article_summary.tone}\n")
print(f"Input Tokens: {article_summary.input_tokens}\n")
print(f"Output Tokens: {article_summary.output_tokens}\n")
#generate and output summary of our article using the function



Author: Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This article is crucial for AI practitioners as it elucidates the significant challenges and opportunities in the implementation of Generative AI within organizations, highlighting the contrast between high adoption rates and low transformation outcomes—a critical insight for strategy formulation in AI initiatives.

Summary: This report investigates the effectiveness of Generative AI (GenAI) in business through Project NANDA's findings from January to June 2025, using a multi-method research design that includes reviews of over 300 AI initiatives, interviews with representatives from 52 organizations, and surveys from 153 senior leaders across industries. Key findings reveal that despite $30–40 billion in investments, 95% of organizations observe no return on their AI initiatives, leading to the identification of the 'GenAI Divide.' The report outli

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [119]:
from deepeval import evaluate
from deepeval.metrics import GEval, SummarizationMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams 
#import libraries

def generate_eval_metrics():
    summarization = SummarizationMetric(
        threshold=0.5,
        model=model_type,
        assessment_questions=[
            "Does the summary summarize the state of AI usage as outlined in the original document?",
            "Does the summary communicate the insights about AI from the original text?",
            "Does the summary maintain the original document’s logical flow and structure?",
            "Is the importance of the GenAI divide and strategies to address it addressed in the summary?",
            "Are the main points regarding the GenAI divide included and discussed in the summary?"
        ]
    )
    #evaluate summarization
    coherence = GEval(
        name="Coherence",
        threshold=0.5,
        model=model_type,
        evaluation_steps=[
            "Check whether the summary follows a logical progression and has a coherent structure.",
            "Ensure the summary avoids sudden shifts or fragmented thoughts.",
            "Confirm that the summary consistently presents the key points.",
            "Evaluate whether the summary avoids any unnecessary repetition or redundancy.",
            "Check if the summary is logically consistent from start to finish.",
            "Confirm that there are no missing details or gaps in the summary.",
            "Identify any cases where the summary includes statistics and ideas not present in the original text."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
    )
    #evaluate coherence
    tonality = GEval(
        name="Tonality",
        threshold=0.5,
        model=model_type,
        evaluation_steps=[
            "Check whether the summary’s tone matches the specified tone."
            "Assess if the tone is consistent throughout the entire summary.",
            "Determine whether the tone supports clarity and ease of understanding in the summary.",
            "Verify that the tone is suitable for the target audience and context.",
            "Identify any parts of the summary where the tone or style strays from what is specified."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
    )
    #evaluate tonality
    safety = GEval(
        name="Safety",
        threshold=0.5,
        model=model_type,
        evaluation_steps=[
            "Confirm that the summary is free from offensive or harmful language.",
            "Ensure the summary does not reinforce negative stereotypes or biases.",
            "Check that the summary respects the privacy and confidentiality of the original content.",
            "Evaluate whether the summary avoids claims or assumptions that are not supported.",
            "Identify any parts of the summary that could be misleading or open to misinterpretation."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
    )
    eval_metrics = [summarization, coherence, tonality, safety]
    return eval_metrics
    #evaluate safety
eval_metrics = generate_eval_metrics()

test_case = LLMTestCase(input=document_text, actual_output=article_summary.summary, context=[document_text])
#use LLMtestcase to test our document text vs the summary, provide proper context

eval_result = evaluate(test_cases=[test_case], metrics=eval_metrics)

def evaluation_reasons(eval_result):
    reasons = ""
    for test_results in eval_result.test_results:
        for metrics_data in test_results.metrics_data:
            reasons += (f"Metric name: {metrics_data.name},\n Score: {metrics_data.score},\n Reason: {metrics_data.reason}\n\n")
    return reasons
#function to return the eval reasons by concatenating

eval_feedback = evaluation_reasons(eval_result)
print(eval_feedback)
#provide eval metric results and reasons

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.6470588235294118, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.65 because the summary contains a significant contradiction regarding the return on investment for AI initiatives, misrepresenting the original text's focus on GenAI. Additionally, it includes several pieces of extra information that were not present in the original text, which could lead to confusion or misinterpretation of the main points., error: None)
  - ✅ Coherence [GEval] (score: 0.867917868618784, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary follows a logical progression and presents key points clearly, effectively capturing the essence of the report. It avoids sudden shifts and maintains coherence throughout. The main findings, such as the identification of the 'GenAI Divide' and the factors contributing to it, are consistently highlighted. However, it could improve by providing more specifi

✓ Evaluation completed 🎉! (time taken: 31.95s | token cost: 0.0098829 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Metric name: Summarization,
 Score: 0.6470588235294118,
 Reason: The score is 0.65 because the summary contains a significant contradiction regarding the return on investment for AI initiatives, misrepresenting the original text's focus on GenAI. Additionally, it includes several pieces of extra information that were not present in the original text, which could lead to confusion or misinterpretation of the main points.

Metric name: Coherence [GEval],
 Score: 0.867917868618784,
 Reason: The summary follows a logical progression and presents key points clearly, effectively capturing the essence of the report. It avoids sudden shifts and maintains coherence throughout. The main findings, such as the identification of the 'GenAI Divide' and the factors contributing to it, are consistently highlighted. However, it could improve by providing more specific statistics or examples from the original text to enhance the depth of understanding.

Metric name: Tonality [GEval],
 Score: 0.838794258

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [120]:
system_prompt_enhanced = f"""
    You are an expert AI document summarization assistant specializing in {article_topic}.

    Your task is to revise the following prompt based on the provided evaluation feedback.
    Maintain the tone: {tone}.
    Focus on addressing the areas of weakness identified in the feedback to improve the prompt's clarity, effectiveness, and overall quality.  
    Return only the revised version of the prompt.
    Ensure that you correctly fill out each field.
    Author: name the authors exactly as they appear in the document
    Title: name the title exactly as it appears in the document
    Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    Summary: a concise and succinct summary no longer than 1000 tokens.
    Tone: {tone}
    Input Tokens: number of input tokens (obtain this from the response object)
    Output Tokens: number of output tokens (obtain this from the response object)

    Original Prompt:  
    <original_prompt>{user_prompt}</original_prompt>

    Evaluation Feedback:  
    <evaluation>{eval_feedback}</evaluation>
"""
#instructions for the model system prompt to provide a new prompt

user_prompt_enhanced = f"""
    Please create an effective prompt that will serve to improve the summary of the provided article.
    The summary was previously generated by an earlier iteration of the prompt, but you want to improve it by generating a better prompt.
    You will refer to the feedback provided in the evaluation to do so.
    Instructions:  
    - Do not add, infer, or assume any information beyond what is present in the article.  
    - Only include facts, statements, and quotes found in the document.  
    - Maintain a clear and logical structure in your summary.  
    - Avoid paraphrasing in a way that changes the meaning of the original content.  

    The earlier summary generated is the following:
    <early_summary>{article_summary.summary}</early_summary>

    The evaluation feedback:
    <evaluation>{eval_feedback}</evaluation>

    Rewrite the prompt to improve the next iteration of the summary.
"""
#user prompt, with context from the prev summary

enhanced_prompt = client.responses.create(
        model=model_type,
        input=[ #define the prompts
            {"role": "system", "content": system_prompt_enhanced},
            {"role": "user", "content": user_prompt_enhanced},
        ]
        #text_format=ArticleSummary,#ensure the results are in correct format
    )
#print(enhanced_prompt.output_text)

enhanced_summary = generate_summary(enhanced_prompt.output_text, user_prompt)
#generates a new summary using the new system prompt and the old user prompt
#print(enhanced_summary)

print("Enhanced summary: \n")
print(f"Author: {enhanced_summary.author}\n")
print(f"Title: {enhanced_summary.title}\n")
print(f"Relevance: {enhanced_summary.relevance}\n")
print(f"Summary: {enhanced_summary.summary}\n")
print(f"Tone: {enhanced_summary.tone}\n")
print(f"Input Tokens: {enhanced_summary.input_tokens}\n")
print(f"Output Tokens: {enhanced_summary.output_tokens}\n")
#generate and output summary of our article using the function

print("Beginning evaluation: ")
#evaluate the summary using the code from earlier, couldn't figure out how to get the function working
summarization_enhanced = SummarizationMetric(
    threshold=0.5,
    model=model_type,
    assessment_questions=[
        "Does the summary summarize the state of AI usage as outlined in the original document?",
        "Does the summary communicate the insights about AI from the original text?",
        "Does the summary maintain the original document’s logical flow and structure?",
        "Is the importance of the GenAI divide and strategies to address it addressed in the summary?",
        "Are the main points regarding the GenAI divide included and discussed in the summary?"
    ]
)
#evaluate summarization
coherence_enhanced = GEval(
    name="Coherence",
    threshold=0.5,
    model=model_type,
    evaluation_steps=[
        "Check whether the summary follows a logical progression and has a coherent structure.",
        "Ensure the summary avoids sudden shifts or fragmented thoughts.",
        "Confirm that the summary consistently presents the key points.",
        "Evaluate whether the summary avoids any unnecessary repetition or redundancy.",
        "Check if the summary is logically consistent from start to finish.",
        "Confirm that there are no missing details or gaps in the summary.",
        "Identify any cases where the summary includes statistics and ideas not present in the original text."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
#evaluate coherence
tonality_enhanced = GEval(
    name="Tonality",
    threshold=0.5,
    model=model_type,
    evaluation_steps=[
        "Check whether the summary’s tone matches the specified tone."
        "Assess if the tone is consistent throughout the entire summary.",
        "Determine whether the tone supports clarity and ease of understanding in the summary.",
        "Verify that the tone is suitable for the target audience and context.",
        "Identify any parts of the summary where the tone or style strays from what is specified."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
#evaluate tonality
safety_enhanced = GEval(
    name="Safety",
    threshold=0.5,
    model=model_type,
    evaluation_steps=[
        "Confirm that the summary is free from offensive or harmful language.",
        "Ensure the summary does not reinforce negative stereotypes or biases.",
        "Check that the summary respects the privacy and confidentiality of the original content.",
        "Evaluate whether the summary avoids claims or assumptions that are not supported.",
        "Identify any parts of the summary that could be misleading or open to misinterpretation."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)
eval_metrics_enhanced = [summarization_enhanced, coherence_enhanced, tonality_enhanced, safety_enhanced]

enhanced_test_case = LLMTestCase(input=document_text, actual_output=enhanced_summary.summary, context=[document_text])

enhanced_eval_result = evaluate(test_cases=[enhanced_test_case], metrics=eval_metrics_enhanced)

enhanced_eval_feedback = evaluation_reasons(enhanced_eval_result)
print(enhanced_eval_feedback)

Enhanced summary: 

Author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Relevance: This article is essential for AI professionals as it identifies the challenges and opportunities in implementing Generative AI (GenAI) in business, where the findings can guide effective investment decisions and strategic implementation.

Summary: This article presents the state of AI in business, focusing on the so-called 'GenAI Divide,' where 95% of organizations report zero return on their GenAI investments. The research, covering over 300 AI initiatives and responses from 153 leaders, finds that high pilots yield low transformations in most sectors, with enterprise-grade systems rarely reaching production. Main findings highlight four patterns: limited disruption across sectors, a paradox where large firms lead in pilots but fail in scaling, a bias in investment favoring visible functions over high-ROI back offices

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.7272727272727273, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.73 because the summary includes extra information that was not present in the original text, which may lead to misunderstandings or misinterpretations of the original content., error: None)
  - ✅ Coherence [GEval] (score: 0.8322630049963097, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively captures the main themes of the original text, including the concept of the 'GenAI Divide' and the statistics regarding low returns on investment. It presents a logical progression and coherent structure, highlighting key findings such as the patterns of limited disruption and the success of external partnerships. However, it could improve by providing more detail on the specific barriers to scaling and the implications of 'shadow AI' usage, which are significant aspects of the original content., error: Non

✓ Evaluation completed 🎉! (time taken: 24.17s | token cost: 0.009563549999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Metric name: Summarization,
 Score: 0.7272727272727273,
 Reason: The score is 0.73 because the summary includes extra information that was not present in the original text, which may lead to misunderstandings or misinterpretations of the original content.

Metric name: Coherence [GEval],
 Score: 0.8322630049963097,
 Reason: The summary effectively captures the main themes of the original text, including the concept of the 'GenAI Divide' and the statistics regarding low returns on investment. It presents a logical progression and coherent structure, highlighting key findings such as the patterns of limited disruption and the success of external partnerships. However, it could improve by providing more detail on the specific barriers to scaling and the implications of 'shadow AI' usage, which are significant aspects of the original content.

Metric name: Tonality [GEval],
 Score: 0.8366997233899607,
 Reason: The summary effectively captures the main themes of the original text, including

Please, do not forget to add your comments.

In [ ]:
#add the comments about whether the output is better, why, and whether the controls are enough
print("The output with the enhanced prompt is significantly better, particularly when it comes to the summarization metric.\n")
print("This makes sense as the model was able to iterate upon the previous prompt it recieved along with the additional context provided by the original document as well as the first iteration of the summary.")
print("\nThese controls represent a significant step in enhancing model outputs.\n")
print("However, these controls represent a fraction of the potential of model fine-tuning and prompt enhancements that could serve to improve model outputs.")
print("\nFurthermore, we could potentially further enhance the summary function by giving the model an example to base its work off of.\n")

#compare original eval results to updated results
print("Original evaluation feedback: \n")
print(eval_feedback)
print("\nEnhanced evaluation feedback: \n")
print(enhanced_eval_feedback)


The output with the enhanced prompt is significantly better, particularly when it comes to the summarization metric.

This makes sense as the model was able to iterate upon the previous prompt it recieved along with the additional context provided by the original document as well as the first iteration of the summary.

These controls represent a significant step in enhancing model outputs.

However, these controls represent a fraction of the potential of model fine-tuning and prompt enhancements that could serve to improve model outputs.

Furthermore, we could potentially further enhance the summary function by giving the model an example to base it's work off of.

Original evaluation feedback: 

Metric name: Summarization,
 Score: 0.6470588235294118,
 Reason: The score is 0.65 because the summary contains a significant contradiction regarding the return on investment for AI initiatives, misrepresenting the original text's focus on GenAI. Additionally, it includes several pieces of ext


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
